In [1]:
#Load Libraries
library(Seurat)
library(Signac)
library(GenomeInfoDb)
library(EnsDb.Hsapiens.v86)
library(BSgenome.Hsapiens.UCSC.hg38)
library(AnnotationHub)
library(GenomicRanges)
library(data.table)
library(dplyr)
library(Matrix)
library(future)

#Set Options
options(future.globals.maxSize = 400000 * 1024^2) #for 400GB max size
plan("sequential")

#Set working directory
setwd("/storage1/fs1/jmillman/Active/DigitalTwin")

Loading required package: SeuratObject

Loading required package: sp

‘SeuratObject’ was built with package ‘Matrix’ 1.7.3 but the current
version is 1.7.4; it is recomended that you reinstall ‘SeuratObject’ as
the ABI for ‘Matrix’ may have changed


Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:base’:

    intersect, t


Loading required package: BiocGenerics

Loading required package: generics


Attaching package: ‘generics’


The following objects are masked from ‘package:base’:

    as.difftime, as.factor, as.ordered, intersect, is.element, setdiff,
    setequal, union



Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, is.unsorted, lapply, Map, mapply, match, mg

# Integrate ATAC Data

In [2]:
atac.combined <- readRDS("checkpoints/DT_ATACmerged.rds")
genome(atac.combined) <- "hg38" # assign genome
atac.combined

An object of class Seurat 
569068 features across 209052 samples within 1 assay 
Active assay: ATAC (569068 features, 142322 variable features)
 2 layers present: counts, data
 2 dimensional reductions calculated: lsi, umap.merged.atac

In [ ]:
#Split timepoints
obj.list <- SplitObject(atac.combined, split.by = "dataset")
obj.list

# Find integration anchors
integration.anchors <- FindIntegrationAnchors(
  object.list = obj.list,
  anchor.features = VariableFeatures(atac.combined),
  reduction = "rlsi",
  dims = 2:30)

saveRDS(integration.anchors, file="checkpoints/DT_ATACanchors.rds")
rm(obj.list)

In [ ]:
integration.anchors <- readRDS("checkpoints/DT_ATACanchors.rds")

# integrate LSI embeddings
integrated.atac <- IntegrateEmbeddings(
  anchorset = integration.anchors,
  reductions = atac.combined[['lsi']],
  new.reduction.name = "integrated_lsi",
  dims.to.integrate = 1:30)

# create a new UMAP using the integrated embeddings
integrated.atac <- RunUMAP(integrated.atac, reduction = "integrated_lsi", dims = 2:30, reduction.name = "umap.integrated.atac")

saveRDS(integrated.atac, file="checkpoints/DT_ATACintegrated.rds")

DefaultAssay(integrated.atac) <- "ATAC"

# Peak Calling using MACS2
peaks <- CallPeaks(integrated.atac, assay="ATAC", macs2.path = "/storage1/fs1/jmillman/Active/Matt/R_libraries/Conda/envs/PeakCalling_analysis/bin/macs2")

# remove peaks on nonstandard chromosomes and in genomic blacklist regions
peaks <- keepStandardChromosomes(peaks, pruning.mode = "coarse")
peaks <- subsetByOverlaps(x = peaks, ranges = blacklist_hg38_unified, invert = TRUE)

# quantify counts in each peak
macs2counts <- FeatureMatrix(fragments = Fragments(integrated.atac), features = peaks, cells = colnames(integrated.atac))

# create a new assay using the MACS2 peak set and add it to the Seurat object
integrated.atac[["peaks"]] <- CreateChromatinAssay(counts = macs2counts, fragments = Fragments(integrated.atac), annotation = annotation)

DefaultAssay(integrated.atac) <- "peaks"
integrated.atac <- FindTopFeatures(integrated.atac, min.cutoff = 5)
integrated.atac <- RunTFIDF(integrated.atac)
integrated.atac <- RunSVD(integrated.atac)

In [ ]:
#saveRDS(integrated.atac, file="checkpoints/DT_ATACintegrated.rds")

In [ ]:
#sink("Notebooks/3_DigitalTwin/04_DT_ATACintegration-sessionInfo.txt")
#sessionInfo()
#sink()